In [1]:
import time

In [4]:
def run_tasks():
    time.sleep(1)
    print("Task 1 done")

    time.sleep(1)
    print("Taksk 2 done")

run_tasks()

Task 1 done
Taksk 2 done


**Execution flow**

```
run_tasks()
    ↓
Task 1 starts
    ↓
time.sleep(1)  ⏳
    ↓
Task 1 done
    ↓
time.sleep(1)  ⏳
    ↓
Task 2 done
```

**Two `sleep()` tasks - Blocking**

In [5]:
def run_tasks():
    time.sleep(2)
    print("Download 1 done")

    time.sleep(1)
    print("Download 2 done")

run_tasks()

Download 1 done
Download 2 done


**Three sequential tasks - Blocking**

In [6]:
def run_tasks():
    time.sleep(1)
    print("Task A done")
    
    time.sleep(1)
    print("Task B done")
    
    time.sleep(1)
    print("Task C done")

run_tasks()

Task A done
Task B done
Task C done


A → wait → B → wait → C

**API calls simulated with `sleep()` - Blocking**

In [7]:
def get_users():
    time.sleep(2)
    print("Users received")

def get_products():
    time.sleep(2)
    print("Products received")

def main():
    get_users()
    get_products()

main()

Users received
Products received


⏱️ Total ≈ 4 seconds

**File processing - Blocking**

In [8]:
def read_file():
    print("Reading file...")
    time.sleep(2)
    print("File read successfully")

def process():
    read_file()
    print("Processing complete")

process()

Reading file...
File read successfully
Processing complete


The program waits for `read_file()` to finish

**User input - Blocking**

In [9]:
def login():
    username = input("Enter username: ")
    password = input("Enter password: ")

    print("Login successful")
login()

Enter username:  john
Enter password:  2345


Login successful


The program waits for:
- Username -> WAIT
- Password -> WAIT

**Asynchronous examples**

**Two async tasks - Non-blocking**

In [15]:
import asyncio

In [16]:
async def task1():
    await asyncio.sleep(2)
    print("Task 1 done")

async def task2():
    await asyncio.sleep(2)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

Task 1 done
Task 2 done


**Three async tasks**

In [17]:
async def task1():
    await asyncio.sleep(2)
    print("Task 1 done")

async def task2():
    await asyncio.sleep(2)
    print("Task 2 done")

async def task3():
    await asyncio.sleep(2)
    print("Task 3 done")

async def main():
    await asyncio.gather(task1(), task2(), task3())

await main()

Task 1 done
Task 2 done
Task 3 done


**Async API requests**

Imagine these represent three API Calls:

In [25]:
import asyncio

async def get_users():
    await asyncio.sleep(2)
    return "users"

async def get_products():
    await asyncio.sleep(3)
    return "products"

async def get_orders():
    await asyncio.sleep(1)
    return "orders"

async def main():
    results = await asyncio.gather(get_users(), get_products(), get_orders())
    print(results)
    
await main()

['users', 'products', 'orders']


⏱️ Total ≈ 3 seconds

Instead of: 2 + 3 + 1 = 6 seconds

they can wait concurrently: Maximum = 3 seconds

Answer: 🟢 Non-blocking

**The dangerous `time.sleep()` inside `async def`**

In [26]:
async def task1():
    time.sleep(2)
    print("Task 1 done")

async def task2():
    time.sleep(2)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

Task 1 done
Task 2 done


You might think:

"It uses async def, so it must be asynchronous."

❌ Wrong.

time.sleep() blocks the event loop.

⏱️ Total ≈ 4 seconds

Answer: 🔴 Blocking

In [33]:
start_time = time.time()

async def task1():
    time.sleep(2)
    print("Task 1 done")

async def task2():
    time.sleep(2)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

end_time = time.time()

elapsed = int(end_time - start_time)
print(f"Elapsed: {elapsed} second(s)")

Task 1 done
Task 2 done
Elapsed: 4 second(s)


**Correct version of dangerous**

In [34]:
async def task1():
    await asyncio.sleep(2)
    print("Task 1 done")

async def task2():
    await asyncio.sleep(2)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

Task 1 done
Task 2 done


In [31]:
start_time = time.time()

async def task1():
    await asyncio.sleep(2)
    print("Task 1 done")

async def task2():
    await asyncio.sleep(2)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

end_time = time.time()

elapsed = int(end_time - start_time)

print(f"Elapsed: {elapsed} second(s)")

Task 1 done
Task 2 done
Elapsed: 2 second(s)


```
▶️ START
   ↓
   Run program
   ↓
   Wait for tasks
   ↓
⏹️ STOP
   ↓
Calculate elapsed time
```

**Sequential `await` - Important**

This looks asynchronous, but the tasks are still executed **one after another.**

In [38]:
start_time = time.time()

async def task1():
    await asyncio.sleep(2)
    print("Task 1 done")

async def task2():
    await asyncio.sleep(2)
    print("Task 2 done")

async def main():
    await task1()
    await task2()

await main()

end_time = time.time()

elapsed = int(end_time - start_time)

print(f"Elapsed: {elapsed} second(s)")

Task 1 done
Task 2 done
Elapsed: 4 second(s)


**Key difference:**

- await task1()
- await task2()

means:
```
Task 1
  ↓
WAIT
  ↓
Task 1 complete
  ↓
Task 2
  ↓
WAIT
  ↓
Task 2 complete
```

Answer: 🟡 Async code, but sequential execution.

This is a very common interview trap.

**`asyncio.gather()` - Concurrent execution**

In [40]:
start_time = time.time()

async def task1():
    await asyncio.sleep(2)
    print("Task 1 done")

async def task2():
    await asyncio.sleep(2)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

end_time = time.time()

elapsed = int(end_time - start_time)

print(f"Elapsed: {elapsed} second(s)")

Task 1 done
Task 2 done
Elapsed: 2 second(s)


⏱️ Total ≈ 2 seconds
```
Task 1 ──────────┐
                 ├── 2 sec
Task 2 ──────────┘
```
Answer: 🟢 Concurrent async execution

**Blocking function moved to a thread**

Suppose you have an old blocking function:

In [43]:
def blocking_work():
    time.sleep(3)
    return "Done"

You can run it from async code using:

In [44]:
async def main():
    results = await asyncio.to_thread(blocking_work)
    print(results)

await main()

Done


The `time.sleep()` is still blocking inside that worker thread, but it doesn't block the main asyncio event loop.

Answer: 🟢 Event loop remains responsive.

**Mixed blocking and non-blocking code**

In [48]:
async def task1():
    await asyncio.sleep(2)
    print("Task A team")

async def task2():
    time.sleep(2)
    print("Task B team")

async def main():
    await asyncio.gather(task1(), task2())

await main()

Task B team
Task A team


Here,

task1 → asyncio.sleep() → 🟢 non-blocking

task2 → time.sleep()    → 🔴 blocking

`task2` blocks the **event loop** while it executes time.sleep().

Answer: 🔴 The program contains a blocking operation.

## 🧠 **Why does Task 2 print first?**

This is the interesting part.

Task 1 says:

**await asyncio.sleep(2)**

Meaning: "Wake me after about 2 seconds, but meanwhile the event loop can do other work."

Task 2 says:

**time.sleep(2)**

Meaning: "Freeze this thread for 2 seconds."

So Task 2 blocks the event loop.

When Task 2 finishes:

```
time.sleep(2)
       ↓
Task 2 done
```
the event loop becomes available again and can resume Task 1.

## 🧠 **The 4 rules to remember**

```
1. time.sleep()
       ↓
   🔴 BLOCKING


2. await asyncio.sleep()
       ↓
   🟢 NON-BLOCKING


3. async def
       ↓
   Does NOT automatically mean non-blocking!


4. asyncio.gather()
       ↓
   Allows multiple async tasks to make progress concurrently
```